In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsevaluleo";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

**### Bronze data Injection**

In [0]:
df_cd = spark.read.csv(f'dbfs:/Volumes/dev/bronze/raw/Cust_det.csv',
  header=True,
  inferSchema=True,
  sep="|")

In [0]:
from pyspark.sql import functions as F

# Add the 'sysdate' column to your existing DataFrame
df_final = df_cd.select(
  F.col("user_id").alias("user_id"),
  F.col("name").alias("name"),
  F.col("phone").alias("phone"),
  F.col("address").alias("address"),
  F.col("country").alias("country"),
  F.current_timestamp().alias("ingestion_date")
)
df_final.write.format("delta") \
  .mode("overwrite").saveAsTable("dev.bronze.cust_detail")

In [0]:
df_Order = spark.read.csv(f'dbfs:/Volumes/dev/bronze/raw/Orders.csv',
  header=True,
  inferSchema=True,
  sep="|")


In [0]:
from pyspark.sql import functions as F

df_final_o = df_Order.select(
  F.col("order_id").cast("string").alias("order_id"),
  F.regexp_replace(F.col("products").cast("string"), " ", "").alias("products")
)
df_final_o.write.format("delta").option("overwriteSchema", "true") \
  .mode("overwrite").saveAsTable("dev.bronze.orders")

In [0]:
df_depart = spark.read.csv(f'dbfs:/Volumes/dev/bronze/raw/deparment.csv',
  header=True,
  inferSchema=True,
  sep="|")

In [0]:
df_final_d = df_depart.select(
  F.col("department_id").cast("string").alias("department_id"),
  F.col("department").alias("department")
)

df_final_d.write.format("delta").option("overwriteSchema", "true") \
  .mode("overwrite").saveAsTable("dev.bronze.Department")

In [0]:
df_order_q = spark.read.csv(f'dbfs:/Volumes/dev/bronze/raw/order_q.csv',
  header=True,
  inferSchema=True,
  sep="|")

In [0]:

df_order_q.write.format("delta").option("overwriteSchema", "true") \
  .mode("overwrite").saveAsTable("dev.bronze.orderq")


In [0]:
df_product_place = spark.read.csv(f'dbfs:/Volumes/dev/bronze/raw/product_place.csv',
  header=True,
  inferSchema=True,
  sep="|")

In [0]:
df_product_place.write.format("delta").option("overwriteSchema", "true") \
  .mode("overwrite").saveAsTable("dev.bronze.product_place")

In [0]:
df_product = spark.read.csv(f'dbfs:/Volumes/dev/bronze/raw/Product.csv',
  header=True,
  inferSchema=True,
  sep="|")

In [0]:
df_product.write.format("delta").option("overwriteSchema", "true") \
  .mode("overwrite").saveAsTable("dev.bronze.Product")

**Silver ETL**

In [0]:
%sql

INSERT OVERWRITE TABLE dev.silver.cust_detail 
SELECT 
    user_id,
    name,
    phone,
    address,
    country,
    ingestion_date
FROM dev.bronze.cust_detail;

In [0]:
%sql
INSERT OVERWRITE TABLE dev.silver.Department 
SELECT 
    CAST(department_id AS INT) AS department_id,
    department
FROM dev.bronze.Department;

In [0]:
%sql
INSERT OVERWRITE TABLE dev.silver.orders 
SELECT 
    cast(order_id as int) as order_id,
    products
FROM dev.bronze.orders;

In [0]:
%sql
INSERT OVERWRITE TABLE dev.silver.orderq 
SELECT 
    CAST(order_id AS INT) AS order_id,
    CAST(user_id AS INT) AS user_id,
    eval_set,
    CAST(order_number AS INT) AS order_number,
    CAST(order_dow AS INT) AS order_dow,
    CAST(order_hour_of_day AS INT) AS order_hour_of_day,
     days_since_prior_order
FROM dev.bronze.orderq;

In [0]:
%sql
INSERT OVERWRITE TABLE dev.silver.Product
SELECT 
    product_id,
    product_name,
    aisle_id,
    department_id
FROM dev.bronze.Product;

In [0]:
%sql
INSERT OVERWRITE TABLE dev.silver.Product_place 
SELECT 
    aisle_id,
    aisle
FROM dev.bronze.Product_place;

In [0]:
%sql
SELECT 
order_id,
order_id as product_id,
product
FROM dev.silver.orders
